# Import

In [2]:
import pandas as pd
import os

# CRISPRCasTyper

## Select good quality MAGs

In [5]:
# (compl >= 70%, cont < 5%)
cmp = 80
cnt = 5

BIG = '/mnt/D050F81950F8084C/Big_data2/PFOWL2_3/Results'

#read tables
mags = pd.read_csv(f'{BIG}/BINs_Dereplicated/mOTUs.tsv', sep='\t', index_col=0, skiprows=5)
qc = pd.read_csv(f'{BIG}/BINs_Dereplicated/mOTUlizer.tsv', sep='\t', index_col=0)

#filter
good = qc.loc[(qc.Completeness >= cmp) & (qc.Contamination < cnt)]
mags = mags.loc[mags.representative.isin(good.index)]
for mag, rep in zip(mags.index, mags.representative):
    mags.loc[mag, ['Completeness','Contamination']] = good.loc[rep, ['Completeness','Contamination']]

    #move
    smpl, fa = rep.split('_', 1)

    !cp $BIG/All_Bins/$smpl/{fa}.fa MAGs/{mag}.fa
    
mags.to_csv('Good_MAGs.csv')

## Run cctyper

In [6]:
OUT = f'{BIG}/cctyper'

!mkdir -p {OUT}

mags = [f for f in os.listdir('MAGs') if f.endswith('.fa')]

for mag in mags:

    !cctyper MAGs/{mag} {OUT}/{mag} --prodigal meta -t 12

/home/ty/miniforge3/envs/cctyper/bin/cctyper:7: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
[2025-04-24 12:01:43] INFO: Running CRISPRCasTyper version 1.8.0
[2025-04-24 12:01:43] INFO: Predicting ORFs with prodigal
[2025-04-24 12:01:47] INFO: Running HMMER against Cas profiles
100%|████████████████████████████████████████| 705/705 [00:05<00:00, 119.21it/s]
[2025-04-24 12:01:53] INFO: Subtyping putative operons
[2025-04-24 12:01:54] INFO: Predicting CRISPR arrays with minced
[2025-04-24 12:01:54] INFO: BLASTing for CRISPR near cas operons
[2025-04-24 12:02:26] INFO: Predicting subtype of CRISPR repeats
[2025-04-24 12:02:26] INFO: Connecting Cas operons and CRISPR arrays
/home/ty/miniforge3/envs/cctyper/lib/python3.11/site-packages/cctyper/crisprcas.py:87: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instea

# Merge spacers

In [7]:
import os

BIG = '/mnt/D050F81950F8084C/Big_data2/PFOWL2_3/Results'
OUT = f'{BIG}/cctyper'

mags = [f for f in os.listdir('MAGs') if f.endswith('.fa')]
with_spacers, without_spacers = 0, 0
with open(f'{OUT}/merged_spacers.fasta', 'w') as merged:
    for mag in mags:
        cctyper = f'{OUT}/{mag}/spacers'
        if not os.path.exists(cctyper):
            without_spacers += 1
            continue
        spacers = [f for f in os.listdir(cctyper) if f.endswith('.fa')]
        with_spacers += 1
        for spa in spacers:
            with open(f'{cctyper}/{spa}', 'r') as fa:
                recs = fa.read().strip('\n>').split('\n>')
                for rec in recs:
                    merged.write(f'>{mag}_{rec}' + '\n')

print(f'Out of {len(mags)} MAGs, {with_spacers} contained spacers and {without_spacers} had none')


Out of 147 MAGs, 65 contained spacers and 82 had none
